In [1]:
import pandas as pd

df = pd.read_csv('../data/customer_support_tickets.csv')
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    2769 non-null   object 
 12  Ticket Priority               8469 non-null   object 
 13  Tic

In [3]:
df['Ticket Status'].value_counts()

Ticket Status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'])
df['Date of Purchase'].min(), df['Date of Purchase'].max()

(Timestamp('2020-01-01 00:00:00'), Timestamp('2021-12-30 00:00:00'))

In [6]:
print(df['Ticket Type'].value_counts())
print()
print(df['Ticket Priority'].value_counts())

Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64

Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64


In [7]:
df['Ticket Channel'].value_counts()

Ticket Channel
Email           2143
Phone           2132
Social media    2121
Chat            2073
Name: count, dtype: int64

In [8]:
df['Ticket Description'].str.len().describe()

count    8469.000000
mean      289.821939
std        43.593954
min       151.000000
25%       273.000000
50%       298.000000
75%       318.000000
max       397.000000
Name: Ticket Description, dtype: float64

In [9]:
df['Resolution'].dropna().str.len().describe()

count    2769.000000
mean       35.798122
std        10.739092
min        11.000000
25%        27.000000
50%        35.000000
75%        44.000000
max        69.000000
Name: Resolution, dtype: float64

In [10]:
import pandas as pd

def crear_documento(row):
    texto = f"""Ticket Type: {row['Ticket Type']}
Subject: {row['Ticket Subject']}
Description: {row['Ticket Description']}
Priority: {row['Ticket Priority']}
Status: {row['Ticket Status']}
Resolution: {row['Resolution'] if pd.notna(row['Resolution']) else 'No resuelto aún'}"""
    
    metadata = {
        'ticket_id': row['Ticket ID'],
        'ticket_type': row['Ticket Type'],
        'priority': row['Ticket Priority'],
        'status': row['Ticket Status'],
        'channel': row['Ticket Channel'],
        'product': row['Product Purchased']
    }
    
    return texto.strip(), metadata

In [12]:
def crear_documento(row):
    # Reemplazar el placeholder {product_purchased} por el producto real
    descripcion = row['Ticket Description'].replace(
        '{product_purchased}', row['Product Purchased']
    )
    
    texto = f"""Ticket Type: {row['Ticket Type']}
Subject: {row['Ticket Subject']}
Description: {descripcion}
Priority: {row['Ticket Priority']}
Status: {row['Ticket Status']}
Resolution: {row['Resolution'] if pd.notna(row['Resolution']) else 'No resuelto aún'}"""
    
    metadata = {
        'ticket_id': row['Ticket ID'],
        'ticket_type': row['Ticket Type'],
        'priority': row['Ticket Priority'],
        'status': row['Ticket Status'],
        'channel': row['Ticket Channel'],
        'product': row['Product Purchased']
    }
    
    return texto.strip(), metadata

In [13]:
documentos = df.apply(crear_documento, axis=1)

# Ver el primer documento como ejemplo
texto_ejemplo, metadata_ejemplo = documentos[0]
print(texto_ejemplo)
print()
print(metadata_ejemplo)

Ticket Type: Technical issue
Subject: Product setup
Description: I'm having an issue with the GoPro Hero. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.
Priority: Critical
Status: Pending Customer Response
Resolution: No resuelto aún

{'ticket_id': 1, 'ticket_type': 'Technical issue', 'priority': 'Critical', 'status': 'Pending Customer Response', 'channel': 'Social media', 'product': 'GoPro Hero'}


In [14]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

print("Todo importó correctamente ✅")

/Users/stars/Documents/Proyecto Alura/Data-Analysis-Agent/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Todo importó correctamente ✅


In [16]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Prueba rápida con un texto de ejemplo
resultado = embeddings.embed_query("Ticket de soporte técnico")
print(f"Embedding generado con {len(resultado)} dimensiones")
print(resultado[:5])  # primeros 5 valores como muestra

Embedding generado con 3072 dimensiones
[-0.015331882983446121, 0.00865185260772705, 0.008678920567035675, -0.061151400208473206, 0.020811300724744797]


In [18]:
from langchain_community.vectorstores import FAISS
from langchain.docstore.document import Document

muestra = df.sample(n=100, random_state=42)

docs = []
for _, row in muestra.iterrows():
    texto, metadata = crear_documento(row)
    docs.append(Document(page_content=texto, metadata=metadata))

print(f"Documentos preparados: {len(docs)}")

vector_store = FAISS.from_documents(docs, embeddings)

print("Vector store creado ✅")

Documentos preparados: 100
Vector store creado ✅


In [19]:
resultados = vector_store.similarity_search("problema con GoPro Hero", k=3)

for r in resultados:
    print(r.page_content[:150])
    print("---")

Ticket Type: Cancellation request
Subject: Product compatibility
Description: I'm having an issue with the GoPro Hero. Please assist.

Please help me.
---
Ticket Type: Refund request
Subject: Battery life
Description: I've noticed a software bug in the GoPro Hero app. It's causing data loss and unexpecte
---
Ticket Type: Technical issue
Subject: Network problem
Description: I'm having an issue with the GoPro Action Camera. Please assist.

Thank you

Mara G
---


In [20]:
vector_store.save_local("../data/vector_store_muestra")
print("Guardado ✅")

Guardado ✅


In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [28]:
UMBRAL_RELEVANCIA = 0.7  

def responder_pregunta(pregunta):
    resultados = vector_store.similarity_search_with_score(pregunta, k=3)
    resultados_relevantes = [(doc, score) for doc, score in resultados if score < UMBRAL_RELEVANCIA]
    
    if not resultados_relevantes:
        return {
            "respuesta": "No encontré información relevante en los tickets disponibles para responder esta pregunta.",
            "fuentes": []
        }
    
    # Arma el contexto con los tickets encontrados
    contexto = "\n\n".join([
        f"[Ticket ID: {doc.metadata['ticket_id']}]\n{doc.page_content}"
        for doc, score in resultados_relevantes
    ])
    
    prompt = f"""Eres un asistente que responde preguntas ÚNICAMENTE basándote en el contexto de tickets de soporte proporcionado abajo. 
No uses conocimiento externo. Si la información no está en el contexto, dilo claramente.
Siempre indica el Ticket ID de donde sacaste cada dato.

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta (indica el Ticket ID de tus fuentes):"""
    
    respuesta = llm.invoke(prompt)
    
    fuentes = [doc.metadata['ticket_id'] for doc, score in resultados_relevantes]
    
    return {
        "respuesta": respuesta.content,
        "fuentes": fuentes
    }

In [29]:
resultado = responder_pregunta("¿Qué problemas ha tenido la gente con GoPro Hero?")
print(resultado["respuesta"])
print()
print("Fuentes (Ticket IDs):", resultado["fuentes"])

La gente ha tenido los siguientes problemas con GoPro Hero:

*   Un error de software en la aplicación GoPro Hero que causa pérdida de datos y errores inesperados (Ticket ID: 3004).
*   Preocupación por la seguridad de los datos en la GoPro Hero (Ticket ID: 3004).
*   Un problema general con la GoPro Hero que no se resolvió con un restablecimiento de fábrica (Ticket ID: 5889).

Fuentes (Ticket IDs): [3004, 5889, 213]


In [30]:
resultado = responder_pregunta("¿Cuál es la capital de Francia?")
print(resultado["respuesta"])
print()
print("Fuentes (Ticket IDs):", resultado["fuentes"])

No encontré información relevante en los tickets disponibles para responder esta pregunta.

Fuentes (Ticket IDs): []
